# RoboPaintBench · Robot painting in Colab

Run the same AgentArk task and SimpleAgent used by the benchmark. Reproduce the Golden Gate reference, or provide your own text/images. **Default: prepare only.** A separate switch starts Unity and paid API calls.

This notebook is a Linux native-Player workflow, not an interactive Unity browser build. The normal hosted preview uses Xvfb/Mesa software rendering; benchmark GPU runs require a separately configured GPU-connected display. Seeing an allocated Colab GPU does not establish that Unity uses it.

Source and runtime versions are locked by `benchmarks/versions.json`; notebook bootstrap tools and Python are pinned below. Transitive Python packages are resolved during installation and their actual versions are saved with the run. The notebook has passed offline validation; a fresh hosted session and paid evaluation still need an end-to-end check.

[Quickstart](https://github.com/P90-RushB/robopaint-bench/blob/main/docs/quickstart.md) · [Custom drawings](https://github.com/P90-RushB/robopaint-bench/blob/main/docs/custom-painting.md) · [Replay](https://github.com/P90-RushB/robopaint-bench/blob/main/docs/replay.md)

Colab resources and session lifetimes vary. Save your outputs before disconnecting. [Colab FAQ](https://research.google.com/colaboratory/faq.html)

In [ ]:
# @title 1. Choose source and create a fresh session
BENCHMARK_REF = "main"  # @param {type:"string"}
LOCAL_BENCHMARK_CHECKOUT = ""  # @param {type:"string"}
# Optional: mounted/local checkout for development or an authorized private copy.
# No credentials are accepted in repository URLs or notebook source.

from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, sys, uuid, json

if sys.platform != "linux" or not Path("/content").is_dir():
    raise RuntimeError("Use a Colab Linux runtime; see quickstart.md for local use.")
SESSION = Path("/content") / ("robopaint-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
SESSION.mkdir(exist_ok=False)
os.environ.update(AGENTARK_AUTO_LOAD_DOTENV="0", AUTO_LOAD_DOTENV="0", GIT_TERMINAL_PROMPT="0")

def run(command, **kwargs):
    # Arguments here contain paths/public versions, never credentials.
    return subprocess.run([str(x) for x in command], check=True, **kwargs)

if LOCAL_BENCHMARK_CHECKOUT.strip():
    BENCH = Path(LOCAL_BENCHMARK_CHECKOUT).expanduser().resolve(strict=True)
else:
    BENCH = SESSION / "benchmark"
    run(["git", "clone", "--filter=blob:none", "https://github.com/P90-RushB/robopaint-bench.git", BENCH])
    run(["git", "-C", BENCH, "checkout", "--detach", BENCHMARK_REF])
if not (BENCH / "benchmarks/versions.json").is_file():
    raise RuntimeError("The selected checkout does not contain RoboPaintBench.")
LOCK = json.loads((BENCH / "benchmarks/versions.json").read_text("utf-8"))
BENCHMARK_COMMIT = subprocess.check_output(["git", "-C", str(BENCH), "rev-parse", "HEAD"], text=True).strip()
BENCHMARK_DIRTY = bool(subprocess.check_output(["git", "-C", str(BENCH), "status", "--porcelain", "--untracked-files=no"], text=True).strip())
print("Session:", SESSION)
print("Benchmark checkout:", BENCHMARK_COMMIT, "tracked changes:", BENCHMARK_DIRTY)
print("AgentArk pin:", LOCK["agentark"]["commit"])
print("Runtime:", LOCK["environment"]["version"], "Task107 task-v1")

## Install dependencies and pinned runtime

The notebook kernel can use a newer Python; evaluation runs in a separate **Python 3.10.12** subprocess environment to satisfy pinned AgentArk's `>=3.10,<3.10.13` constraint. No kernel replacement or host Python downgrade is needed. This setup intentionally creates a fresh environment inside this disposable notebook session.

The two locked Linux ZIPs total about 145 MB compressed, plus Python/packages. The installer checks immutable revision, size and SHA-256 and refuses existing destinations. A private GitHub repository/dataset requires your own access; this notebook does not grant it. Do not paste credentials into clone URLs.

[uv Python installation](https://docs.astral.sh/uv/concepts/python-versions/) · [Hugging Face versioned download](https://huggingface.co/docs/huggingface_hub/en/guides/download)

In [ ]:
# @title 2. Install the Python subprocess environment and graphics libraries
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "git", "xvfb", "xauth", "mesa-utils", "libgl1", "libgl1-mesa-dri", "libglib2.0-0", "libxrandr2", "libxcursor1", "libxi6", "libxinerama1", "libpulse0", "libnss3", "ffmpeg"])
run([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.7"])
run([sys.executable, "-m", "uv", "python", "install", "3.10.12", "--no-bin"])
VENV = SESSION / "python-env"
run([sys.executable, "-m", "uv", "venv", "--python", "3.10.12", "--seed", VENV])
PYTHON = VENV / "bin/python"
AGENTARK = SESSION / "AgentArk"
run(["git", "clone", "--filter=blob:none", "https://github.com/P90-RushB/AgentArk.git", AGENTARK])
run(["git", "-C", AGENTARK, "checkout", "--detach", LOCK["agentark"]["commit"]])
run([PYTHON, "-m", "pip", "install", "-q", "-e", AGENTARK, "huggingface_hub==0.36.0"])
run([PYTHON, "-c", "import sys; assert sys.version_info[:3] == (3, 10, 12); import agent_ark; print('Python:', sys.version.split()[0]); print('AgentArk import:', agent_ark.__file__)"])
print("Dependencies installed; Unity and model calls have not started.")

In [ ]:
# @title 3. Optional dataset authentication and verified runtime installation
HF_AUTH_REQUIRED = False  # @param {type:"boolean"}
# Leave false for public downloads. If access is required, enable it and use
# Colab Secret HF_TOKEN or a hidden prompt. The token is not written to a file.

import getpass

def secret(name, required=True):
    value = os.environ.get(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            pass  # Do not print secret-service exception bodies.
    if not value and required:
        value = getpass.getpass(name + " (hidden): ").strip()
    if required and not value:
        raise RuntimeError("Missing credential: " + name)
    if value:
        os.environ[name] = value
    return bool(value)

if HF_AUTH_REQUIRED:
    secret("HF_TOKEN")
PLAYER = SESSION / "player"
run([PYTHON, BENCH / "scripts/install_runtime.py", "--platform", "linux", "--destination", PLAYER])
MODS = PLAYER / "AgentArk_Data/Resources/Mods"
os.environ.update(AGENTARK_ENV_PATH=str(PLAYER / "AgentArk.x86_64"), AGENTARK_MOD_PATH=str(MODS), AGENTARK_TASK_STORE_PATH=str(MODS / "all_tasks"))
print("Verified runtime ready. No model API key has been requested.")

In [ ]:
# @title 4. Select drawing, model and rendering profile
TARGET_MODE = "golden_gate"  # @param ["golden_gate", "custom_text", "custom_images", "pelican_template"]
MODEL = "sol"  # @param ["sol", "argon", "opus", "grok", "wonder", "deepseek", "qwen", "kimi", "mimo", "doubao"]
CUSTOM_GOAL = "Paint a small red lighthouse on a rocky island at sunset."  # @param {type:"string"}
RENDER_MODE = "xvfb-software"  # @param ["xvfb-software", "existing-display"]
RECORD_MOTION_VIDEO = False  # @param {type:"boolean"}

# existing-display requires a configured GPU-connected DISPLAY. A hosted Colab
# GPU allocation alone is insufficient. Xvfb software preview is the default.
# Astra's historical subscription transport requires separately reviewed
# compatibility adapters; see quickstart.md. No substitute profile is implied.
if TARGET_MODE.startswith("custom_") and not CUSTOM_GOAL.strip():
    raise ValueError("A custom drawing needs nonempty CUSTOM_GOAL.")
if TARGET_MODE == "pelican_template":
    print("Pelican template selected; independent benchmark results are pending integration.")
print("Selected:", TARGET_MODE, MODEL, RENDER_MODE)

def input_identity(goal_bytes, reference_paths):
    import hashlib
    references = []
    for path in reference_paths:
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        references.append({"basename": path.name, "bytes": path.stat().st_size, "sha256": digest.hexdigest()})
    return {"goal_sha256": hashlib.sha256(goal_bytes).hexdigest(), "references": references}


def selected_input_identity():
    case_name = "pelican" if TARGET_MODE == "pelican_template" else "golden_gate"
    goal_bytes = ((CUSTOM_GOAL + "\n").encode("utf-8") if TARGET_MODE.startswith("custom_")
                  else (BENCH / "benchmarks" / case_name / "painting_goal.txt").read_bytes())
    if TARGET_MODE == "custom_images":
        references = list(UPLOADED_REFERENCES)
    elif TARGET_MODE in ("custom_text", "pelican_template"):
        references = []
    else:
        references = [MODS / "all_tasks/RobotPainting/References" / name for name in LOCK["reference_images"]]
    return input_identity(goal_bytes, references)


In [ ]:
# @title 5. Upload custom reference images, if selected
UPLOADED_REFERENCES = []
if TARGET_MODE == "custom_images":
    from google.colab import files
    UPLOADS = SESSION / "uploads"
    UPLOADS.mkdir(exist_ok=False)
    old_cwd = Path.cwd()
    try:
        os.chdir(UPLOADS)
        uploaded = files.upload()
    finally:
        os.chdir(old_cwd)
    for filename in uploaded:
        if Path(filename).name != filename:
            raise ValueError("Use a simple safe image basename.")
        UPLOADED_REFERENCES.append(UPLOADS / filename)
    if not UPLOADED_REFERENCES:
        raise ValueError("Upload at least one PNG/JPEG reference.")
    print("Reference order:", [p.name for p in UPLOADED_REFERENCES])
    print("Describe image 1, image 2, etc. in CUSTOM_GOAL. Reorder the path list here if needed.")
else:
    print("No custom reference upload required.")

In [ ]:
# @title 6. Prepare only — no Unity or paid inference
selected_inputs = selected_input_identity()
RUN = SESSION / ("run-" + uuid.uuid4().hex[:8])
case = "pelican" if TARGET_MODE == "pelican_template" else "golden_gate"
command = [PYTHON, BENCH / "scripts/prepare_run.py", "--case", case, "--model", MODEL, "--runtime-platform", "linux", "--render-mode", RENDER_MODE, "--output-dir", RUN]
if TARGET_MODE.startswith("custom_"):
    goal = SESSION / "custom_goal.txt"
    goal.write_text(CUSTOM_GOAL + "\n", encoding="utf-8")
    command.extend(["--goal-file", goal])
    if TARGET_MODE == "custom_text":
        command.append("--text-only")
    else:
        for image in UPLOADED_REFERENCES:
            command.extend(["--reference-image", image])
run(command, cwd=BENCH)
CONFIG = RUN / "eval.yaml"
manifest = json.loads((RUN / "manifest.json").read_text("utf-8"))
if (manifest["goal_sha256"] != selected_inputs["goal_sha256"]
        or manifest["reference_sha256"] != [item["sha256"] for item in selected_inputs["references"]]):
    raise RuntimeError("Inputs changed during preparation. Prepare a fresh run first.")
freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
(RUN / "python-packages.txt").write_text(freeze, encoding="utf-8")
(RUN / "notebook_runtime.json").write_text(json.dumps({"benchmark_commit": BENCHMARK_COMMIT, "benchmark_tracked_source_dirty": BENCHMARK_DIRTY, "python": "3.10.12", "render_mode": RENDER_MODE, "target_mode": TARGET_MODE, "selected_inputs": selected_inputs, "agentark_commit": LOCK["agentark"]["commit"], "hosted_execution_verified_by_public_notebook_release": False}, indent=2) + "\n", encoding="utf-8")
print("Prepared:", CONFIG)
print("Protocol:", manifest["protocol"])
print("Custom target:", manifest["custom_target"])
print("Render declaration:", manifest["rendering"])
print("Inspect manifest.json and effective_task.preview.yaml in the Colab file browser before launch.")

## Start explicitly

The next cell can call the selected provider and incur API charges. It runs up to three attempts × 96 environment actions, with possible additional model requests for tools and summaries. Runtime and token cost can be substantial; no price estimate or fixed completion time is promised. Runtime credentials are read only when starting.

`START_EVALUATION` stays false when running all cells by default. Set it true only when ready, then run the cell. Do not rerun the same launch into the same output directory. Changing the goal text, reference contents/order or target/model requires a fresh preparation; the start cell verifies hashes before requesting credentials.

In [ ]:
# @title 7. Explicit start switch (default off)
START_EVALUATION = False  # @param {type:"boolean"}

if not START_EVALUATION:
    print("Prepared only. Toggle START_EVALUATION to true to launch.")
else:
    if (RUN / "evaluation_started.json").exists():
        raise RuntimeError("This run already started. Prepare a new unique run; do not overwrite its records.")
    prepared_runtime = json.loads((RUN / "notebook_runtime.json").read_text("utf-8"))
    if (manifest["model"]["name"] != MODEL or prepared_runtime["target_mode"] != TARGET_MODE
            or prepared_runtime["render_mode"] != RENDER_MODE):
        raise RuntimeError("Selections changed after preparation. Prepare a fresh run first.")
    if prepared_runtime["selected_inputs"] != selected_input_identity():
        raise RuntimeError("Drawing inputs changed after preparation. Prepare a fresh run first.")
    copied_inputs = input_identity((RUN / "inputs/painting_goal.txt").read_bytes(),
                                  [RUN / "inputs" / item["basename"] for item in prepared_runtime["selected_inputs"]["references"]])
    if copied_inputs != prepared_runtime["selected_inputs"]:
        raise RuntimeError("Prepared input copies changed. Prepare a fresh run first.")
    child_env = os.environ.copy()
    for override in ("AGENTARK_TASK107_CANVAS_OUTPUT", "AGENTARK_CANVAS_OUTPUT_DIRECTORY"):
        child_env.pop(override, None)
    if RENDER_MODE == "xvfb-software":
        child_env.update(LIBGL_ALWAYS_SOFTWARE="1", GALLIUM_DRIVER="llvmpipe", GALLIUM_NUM_THREADS="2", LP_NUM_THREADS="2")
        preflight_command = ["xvfb-run", "-a", "-s", "-screen 0 1920x1080x24 +extension GLX +render -noreset", "glxinfo", "-B"]
    else:
        if not child_env.get("DISPLAY"):
            raise RuntimeError("existing-display needs a configured GPU-connected DISPLAY.")
        preflight_command = ["glxinfo", "-B"]
    graphics = subprocess.run(preflight_command, env=child_env, capture_output=True, text=True, check=True)
    (RUN / "render_preflight.txt").write_text(graphics.stdout, encoding="utf-8")
    if RENDER_MODE == "existing-display" and any(term in graphics.stdout.lower() for term in ("llvmpipe", "softpipe", "swrast", "software rasterizer")):
        raise RuntimeError("A software GL renderer was found. Select preview mode or configure the GPU display.")
    for line in graphics.stdout.splitlines():
        if "renderer" in line.lower():
            print(line)
    # Preflight checks GLX. Confirm Unity's actual renderer separately in player.log.
    key_name = "ARK_API_KEY" if MODEL == "doubao" else "OPENROUTER_API_KEY"
    secret(key_name)
    child_env[key_name] = os.environ[key_name]
    if RECORD_MOTION_VIDEO:
        import shutil
        ffmpeg = shutil.which("ffmpeg")
        if not ffmpeg:
            raise RuntimeError("No ffmpeg executable found.")
        child_env.update(AGENTARK_TASK107_VIDEO_DIR=str(RUN / "videos"), AGENTARK_FFMPEG=ffmpeg)
    else:
        child_env.pop("AGENTARK_TASK107_VIDEO_DIR", None)
    (RUN / "evaluation_started.json").write_text(json.dumps({"started_utc": datetime.now(timezone.utc).isoformat(), "render_mode": RENDER_MODE, "motion_video": RECORD_MOTION_VIDEO}) + "\n", encoding="utf-8")
    evaluation_log = (RUN / "evaluation.log").open("x", encoding="utf-8")
    evaluation_process = subprocess.Popen([str(PYTHON), str(BENCH / "scripts/run_eval.py"), "--config", str(CONFIG)], cwd=BENCH, env=child_env, stdout=evaluation_log, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    evaluation_log.close()
    print("Evaluation started. PID:", evaluation_process.pid)
    print("Local log:", RUN / "evaluation.log")
    print("Unity graphics log:", RUN / "player.log")

In [ ]:
# @title 8. Status and saved canvas preview (rerun while evaluation proceeds)
if "evaluation_process" in globals():
    status = evaluation_process.poll()
    print("Evaluation running" if status is None else "Evaluation exit code: " + str(status))
else:
    print("No evaluation has been launched in this notebook session.")

from IPython.display import display, Image
canvases = sorted((RUN / "paintings").glob("**/canvas.png"))
for canvas in canvases[-3:]:
    print(canvas.parent.name)
    display(Image(filename=str(canvas), width=640))
if not canvases:
    print("No saved canvas yet; preparation does not generate a painting.")
print("Episode directory counters are not guaranteed to be attempt numbers; use metadata/episode GUIDs for mapping.")

In [ ]:
# @title 9. Optional stop or download (both default off)
STOP_EVALUATION = False  # @param {type:"boolean"}
DOWNLOAD_RUN = False  # @param {type:"boolean"}

if STOP_EVALUATION and "evaluation_process" in globals() and evaluation_process.poll() is None:
    import signal
    os.killpg(evaluation_process.pid, signal.SIGTERM)
    print("Stop requested. Wait for process exit and check for partial artifacts; this is not a complete benchmark result.")
if DOWNLOAD_RUN:
    if "evaluation_process" in globals() and evaluation_process.poll() is None:
        raise RuntimeError("Wait for the evaluation process to exit before exporting.")
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(SESSION / (RUN.name + "-export")), "zip", root_dir=RUN)
    files.download(archive)
    print("The archive may be large and contain model responses and observation images. Review it before sharing.")

## Replay, videos and release status

Use [the replay guide](https://github.com/P90-RushB/robopaint-bench/blob/main/docs/replay.md) with the complete action record and original inputs. This requires the Unity runtime but no model API calls. Saved canvases are presentation artifacts; reward measures the submission workflow, not artistic quality.

Motion recording is optional and uses the task's existing 1920×1080/12fps recorder. It records simulated drawing motion and omits model-thinking waits. Human video cameras are separate from model observations. Check recorder completion and decoded frames before publishing. Bilibili/YouTube links and the independent Pelican result set will be added after they are available.

Before sharing a notebook copy, clear its outputs and inspect source cells for private paths/prompts. This released notebook contains no stored outputs or credentials. Runtime `runs`, raw records, media and API bodies are excluded from source Git by default.